# Ping-Pong ASCII
Execute a célula, clique no desenho. O jogo começa automaticamente. Use as setas ou botões.

Python exibe HTML; JavaScript controla o jogo no navegador. Não precisa instalar pacotes.

A cobrinha atravessa as bordas e o próprio corpo. O Tetris usa blocos ASCII, o Pong é contra o computador e o RPG pede uma chave para abrir a porta.

In [ ]:
from IPython.display import Javascript, display

# Execute esta célula inteira. Clique no desenho e use as setas.
# Python exibe a saída; JavaScript controla o jogo no navegador.
display(Javascript(r"""(() => {
// O painel fica na saída desta célula, sem iframe dentro de iframe.
const painel = document.createElement('div');
painel.style = 'background:#111;color:#a6ff99;padding:12px;max-width:100%;font:14px monospace';
painel.innerHTML = "<b>Ping-Pong ASCII</b><p><button data-action=\"restart\">Reiniciar</button> <button data-action=\"pause\">Pausar / continuar</button></p><span role=\"status\"></span><pre tabindex=\"0\" aria-label=\"Jogo ASCII: use as setas\" style=\"font:14px/1.2 monospace;overflow:auto;outline:1px solid #486;padding:6px\"></pre><div><button data-key=\"ArrowUp\">Cima</button><button data-key=\"ArrowDown\">Baixo</button><button data-key=\"ArrowLeft\">Esquerda</button><button data-key=\"ArrowRight\">Direita</button></div>";
document.body.appendChild(painel);
const tela = painel.querySelector('pre'), aviso = painel.querySelector('[role="status"]');
let tempo = null, rodando = false, terminou = false;
function moldura(linhas) {
    const borda = '+' + '-'.repeat(linhas[0].length) + '+';
    return [borda, ...linhas.map(l => '|' + l + '|'), borda].join(String.fromCharCode(10));
}
// A bola O rebate nas raquetes |. Vença ao marcar 5 pontos.
const largura = 28, altura = 12;
let jogador, computador, bolaX, bolaY, vx, vy, pontos, rival, passos;
function servir(sentido) {
    bolaX = Math.floor(largura / 2); bolaY = Math.floor(altura / 2);
    vx = sentido; vy = 1;
}
function reiniciar() {
    jogador = computador = 5; pontos = rival = passos = 0; servir(1);
}
function comando(tecla) {
    if (tecla === 'ArrowUp') jogador = Math.max(1, jogador - 1);
    if (tecla === 'ArrowDown') jogador = Math.min(altura - 2, jogador + 1);
}
function atualizar() {
    passos++;
    if (passos % 2 === 0) computador = Math.max(1, Math.min(altura - 2, computador + Math.sign(bolaY - computador)));
    bolaX += vx; bolaY += vy;
    if (bolaY <= 0 || bolaY >= altura - 1) vy = -vy;
    if (bolaX === 1 && vx < 0 && Math.abs(bolaY - jogador) <= 1) vx = 1;
    if (bolaX === largura - 2 && vx > 0 && Math.abs(bolaY - computador) <= 1) vx = -1;
    if (bolaX < 0) { rival++; servir(1); }
    if (bolaX >= largura) { pontos++; servir(-1); }
    terminou = pontos >= 5 || rival >= 5;
}
function desenhar() {
    const linhas = [];
    for (let y = 0; y < altura; y++) {
        let linha = '';
        for (let x = 0; x < largura; x++) {
            if (x === bolaX && y === bolaY) linha += 'O';
            else if ((x === 1 && Math.abs(y-jogador) <= 1) || (x === largura-2 && Math.abs(y-computador) <= 1)) linha += '|';
            else linha += x === Math.floor(largura/2) ? ':' : ' ';
        }
        linhas.push(linha);
    }
    return moldura(linhas) + '\nVoce: ' + pontos + ' | Computador: ' + rival + '\n' + (terminou ? (pontos >= 5 ? 'VOCE VENCEU!' : 'FIM DE JOGO. Tente novamente!') : 'Setas para cima/baixo: mover a raquete');
}

function parar() { clearInterval(tempo); tempo = null; rodando = false; }
function mostrar() {
    tela.textContent = desenhar();
    if (terminou) { parar(); aviso.textContent = 'Fim da partida. Use Reiniciar.'; }
}
function continuar() {
    if (terminou) return;
    parar(); rodando = true;
    aviso.textContent = 'Clique no desenho e use as setas, ou toque nos botoes.';
    tempo = setInterval(() => { if (!painel.isConnected) { parar(); return; } atualizar(); mostrar(); }, 160);
    tela.focus();
}
function iniciar() { parar(); terminou = false; reiniciar(); mostrar(); continuar(); }
function receber(tecla) { if (rodando && !terminou) { comando(tecla); mostrar(); } }
tela.onkeydown = e => {
    if (['ArrowUp','ArrowDown','ArrowLeft','ArrowRight',' '].includes(e.key)) {
        e.preventDefault(); receber(e.key);
    }
};
painel.querySelectorAll('[data-key]').forEach(b => b.onclick = () => { receber(b.dataset.key); tela.focus(); });
painel.querySelector('[data-action="restart"]').onclick = iniciar;
painel.querySelector('[data-action="pause"]').onclick = () => {
    if (rodando) { parar(); aviso.textContent = 'Pausado.'; } else continuar();
};
window.addEventListener('pagehide', parar, {once:true});
iniciar();
})();"""))